# Part 3 – Advanced Modeling

In this notebook, i build advanced machine learning models for classification.

The objectives are:

- Decision Tree Classifier
- Random Forest Classifier
- Gradient Boosting Classifier
- Cross Validation
- Hyperparameter Tuning
- Machine Learning Pipeline
- Model Serialization
- Model Comparison

Dataset used:
cleaned_data.csv

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve
)

import joblib

In [2]:
df = pd.read_csv("../data/cleaned_data.csv")

df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
X = df.drop("SalePrice", axis=1)

y_reg = df["SalePrice"]

y_clf = (y_reg > y_reg.median()).astype(int)

In [4]:
X = pd.get_dummies(
    X,
    drop_first=True
)

print("Shape :", X.shape)

Shape : (1460, 245)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# Decision Tree Classifier (Baseline Model)

In this task, we train a Decision Tree Classifier using the default parameters.

The model is evaluated using both training accuracy and testing accuracy to check whether the model is overfitting.

In [7]:
dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(
    X_train_scaled,
    y_train
)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [8]:
train_pred = dt_model.predict(X_train_scaled)

test_pred = dt_model.predict(X_test_scaled)

In [9]:
train_accuracy = accuracy_score(
    y_train,
    train_pred
)

test_accuracy = accuracy_score(
    y_test,
    test_pred
)

print("Training Accuracy :", train_accuracy)
print("Testing Accuracy :", test_accuracy)

Training Accuracy : 1.0
Testing Accuracy : 0.8732876712328768


In [10]:
comparison = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Accuracy": [train_accuracy, test_accuracy]
})

comparison

,Dataset,Accuracy
0,Training,1.000000
1,Testing,0.873288


### Observation

The Decision Tree achieved 100% training accuracy but a lower testing accuracy of 87.33%.

This indicates that the model has overfitted the training data. It has learned the training samples very well but does not generalize equally well on unseen test data.

# Controlled Decision Tree

In this task, we reduce overfitting by limiting the depth of the Decision Tree and controlling the minimum number of samples required for splitting a node.

In [11]:
controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

controlled_tree.fit(
    X_train_scaled,
    y_train
)

,criterion,'gini'
,splitter,'best'
,max_depth,5
,min_samples_split,20
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [12]:
train_pred_controlled = controlled_tree.predict(X_train_scaled)

test_pred_controlled = controlled_tree.predict(X_test_scaled)

In [13]:
train_accuracy_controlled = accuracy_score(
    y_train,
    train_pred_controlled
)

test_accuracy_controlled = accuracy_score(
    y_test,
    test_pred_controlled
)

print("Training Accuracy :", train_accuracy_controlled)

print("Testing Accuracy :", test_accuracy_controlled)

Training Accuracy : 0.9383561643835616
Testing Accuracy : 0.9041095890410958


In [14]:
comparison = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Accuracy": [
        train_accuracy_controlled,
        test_accuracy_controlled
    ]
})

comparison

,Dataset,Accuracy
0,Training,0.938356
1,Testing,0.904110


### Observation

The controlled Decision Tree reduced the gap between training accuracy and testing accuracy.

Compared to the baseline model, the testing accuracy improved while the training accuracy decreased slightly. This indicates that limiting the tree depth helped reduce overfitting and improved the model's ability to generalize on unseen data.

# Gini and Entropy Comparison

In this task, two Decision Tree models are trained using different splitting criteria: Gini Impurity and Entropy.

The performance of both models is compared using testing accuracy.

In [15]:
gini_tree = DecisionTreeClassifier(
    criterion="gini",
    max_depth=5,
    random_state=42
)

entropy_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,
    random_state=42
)

In [16]:
gini_tree.fit(X_train_scaled, y_train)

entropy_tree.fit(X_train_scaled, y_train)

,criterion,'entropy'
,splitter,'best'
,max_depth,5
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [17]:
gini_prediction = gini_tree.predict(X_test_scaled)

entropy_prediction = entropy_tree.predict(X_test_scaled)

In [18]:
gini_accuracy = accuracy_score(
    y_test,
    gini_prediction
)

entropy_accuracy = accuracy_score(
    y_test,
    entropy_prediction
)

print("Gini Accuracy :", gini_accuracy)

print("Entropy Accuracy :", entropy_accuracy)

Gini Accuracy : 0.8938356164383562
Entropy Accuracy : 0.9143835616438356


In [19]:
comparison = pd.DataFrame({
    "Criterion": ["Gini", "Entropy"],
    "Testing Accuracy": [
        gini_accuracy,
        entropy_accuracy
    ]
})

comparison

,Criterion,Testing Accuracy
0,Gini,0.893836
1,Entropy,0.914384


### Observation

Both Gini and Entropy produced good classification performance.

In this dataset, the Decision Tree using Entropy achieved slightly higher testing accuracy than the Decision Tree using Gini Impurity.

# Random Forest Classifier

In this task, a Random Forest Classifier is trained to improve prediction performance by combining multiple Decision Trees.

The model is evaluated using training accuracy, testing accuracy, ROC-AUC score, and feature importance.

In [20]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_model.fit(
    X_train_scaled,
    y_train
)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [21]:
train_pred_rf = rf_model.predict(X_train_scaled)

test_pred_rf = rf_model.predict(X_test_scaled)

test_prob_rf = rf_model.predict_proba(
    X_test_scaled
)[:, 1]

In [22]:
train_accuracy_rf = accuracy_score(
    y_train,
    train_pred_rf
)

test_accuracy_rf = accuracy_score(
    y_test,
    test_pred_rf
)

auc_rf = roc_auc_score(
    y_test,
    test_prob_rf
)

print("Training Accuracy :", train_accuracy_rf)

print("Testing Accuracy :", test_accuracy_rf)

print("ROC-AUC Score :", auc_rf)

Training Accuracy : 0.997431506849315
Testing Accuracy : 0.928082191780822
ROC-AUC Score : 0.9795927941452429


In [23]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(5)

,Feature,Importance
16,GrLivArea,0.067951
6,YearBuilt,0.059878
19,FullBath,0.055129
4,OverallQual,0.054327
26,GarageCars,0.051691


### Observation

The Random Forest model achieved very high training accuracy and strong testing accuracy.

The ROC-AUC score is also very high, indicating that the model can effectively distinguish between the two classes.

The feature importance scores show that GrLivArea, YearBuilt, FullBath, OverallQual, and GarageCars contribute the most to the prediction.

# Gradient Boosting Classifier

In this task, a Gradient Boosting Classifier is trained and evaluated using training accuracy, testing accuracy, and ROC-AUC score.

In [24]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gb_model.fit(
    X_train_scaled,
    y_train
)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [25]:
train_pred_gb = gb_model.predict(X_train_scaled)

test_pred_gb = gb_model.predict(X_test_scaled)

test_prob_gb = gb_model.predict_proba(
    X_test_scaled
)[:, 1]

In [26]:
train_accuracy_gb = accuracy_score(
    y_train,
    train_pred_gb
)

test_accuracy_gb = accuracy_score(
    y_test,
    test_pred_gb
)

auc_gb = roc_auc_score(
    y_test,
    test_prob_gb
)

print("Training Accuracy :", train_accuracy_gb)

print("Testing Accuracy :", test_accuracy_gb)

print("ROC-AUC Score :", auc_gb)

Training Accuracy : 0.9931506849315068
Testing Accuracy : 0.934931506849315
ROC-AUC Score : 0.981610058172265


### Observation

The Gradient Boosting model achieved the highest testing accuracy and ROC-AUC score among the models evaluated so far.

This indicates that Gradient Boosting provides better generalization and class separation than both the Decision Tree and Random Forest models on this dataset.

# Feature Ablation Study

In this task, the five least important features identified by the Random Forest model are removed. A new Random Forest model is then trained and compared with the original model using ROC-AUC score.

In [27]:
lowest_features = feature_importance.sort_values(
    by="Importance"
).head(5)

print("Lowest 5 Important Features")

lowest_features

Lowest 5 Important Features


,Feature,Importance
91,Condition2_PosN,0.0
57,Neighborhood_BrDale,0.0
94,Condition2_RRNn,0.0
231,MiscFeature_TenC,0.0
140,Exterior2nd_Other,0.0


In [28]:
X_reduced = X.drop(
    columns=lowest_features["Feature"]
)

In [29]:
X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = train_test_split(
    X_reduced,
    y_clf,
    test_size=0.2,
    random_state=42,
    stratify=y_clf
)

In [30]:
scaler = StandardScaler()

X_train_reduced_scaled = scaler.fit_transform(
    X_train_reduced
)

X_test_reduced_scaled = scaler.transform(
    X_test_reduced
)

In [31]:
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

rf_reduced.fit(
    X_train_reduced_scaled,
    y_train_reduced
)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [32]:
reduced_probability = rf_reduced.predict_proba(
    X_test_reduced_scaled
)[:, 1]

reduced_auc = roc_auc_score(
    y_test_reduced,
    reduced_probability
)

comparison = pd.DataFrame({
    "Model": [
        "Full Random Forest",
        "Reduced Random Forest"
    ],
    "ROC-AUC": [
        auc_rf,
        reduced_auc
    ]
})

comparison

,Model,ROC-AUC
0,Full Random Forest,0.979593
1,Reduced Random Forest,0.977013


### Observation

The reduced Random Forest model achieved a ROC-AUC score very close to the full model.

Removing the five least important features caused only a very small decrease in performance, suggesting that these features contributed very little to the prediction.

# Cross-Validated Model Comparison

In this section, four classification models are evaluated using 5-fold Stratified Cross Validation.

The mean ROC-AUC score and standard deviation are calculated for each model to compare their overall performance and stability.

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

In [34]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [35]:
logistic_cv = LogisticRegression(
    max_iter=1000,
    random_state=42
)

decision_tree_cv = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

random_forest_cv = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

gradient_boosting_cv = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

In [36]:
models = {
    "Logistic Regression": logistic_cv,
    "Decision Tree": decision_tree_cv,
    "Random Forest": random_forest_cv,
    "Gradient Boosting": gradient_boosting_cv
}

In [37]:
results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=cv,
        scoring="roc_auc"
    )

    results.append({
        "Model": name,
        "Mean AUC": scores.mean(),
        "Standard Deviation": scores.std()
    })

cv_results = pd.DataFrame(results)

cv_results

,Model,Mean AUC,Standard Deviation
0,Logistic Regression,0.951444,0.012478
1,Decision Tree,0.929551,0.010724
2,Random Forest,0.980716,0.005626
3,Gradient Boosting,0.978707,0.007252


### Observation

Random Forest achieved the highest mean ROC-AUC score with the lowest standard deviation among all models.

This indicates that the Random Forest model provides both high predictive performance and consistent results across different folds of the training data.

# Hyperparameter Tuning using GridSearchCV

In this task, a machine learning pipeline is created using a Simple Imputer, StandardScaler, and Random Forest Classifier.

GridSearchCV is used to find the best combination of hyperparameters using 5-fold Stratified Cross Validation.

In [38]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV

In [39]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)

pipeline

,steps,"[('simpleimputer', ...), ('standardscaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


In [40]:
param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

param_grid

{'randomforestclassifier__n_estimators': [50, 100, 200],
 'randomforestclassifier__max_depth': [5, 10, None],
 'randomforestclassifier__min_samples_leaf': [1, 5]}

In [41]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    ),
    scoring="roc_auc",
    n_jobs=-1
)

In [42]:
grid_search.fit(
    X_train,
    y_train
)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'randomforestclassifier__max_depth': [5, 10, ...], 'randomforestclassifier__min_samples_leaf': [1, 5], 'randomforestclassifier__n_estimators': [50, 100, ...]}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,missing_values,nan


In [43]:
print("Best Parameters")

print(grid_search.best_params_)

print()

print("Best ROC-AUC Score")

print(grid_search.best_score_)

Best Parameters
{'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best ROC-AUC Score
0.9812036387003792


In [44]:
best_pipeline = grid_search.best_estimator_

best_pipeline

,steps,"[('simpleimputer', ...), ('standardscaler', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,copy,True


### Observation

GridSearchCV evaluated different combinations of Random Forest hyperparameters using 5-fold cross validation.

The best model used 200 trees, a maximum depth of 10, and a minimum leaf size of 1. This configuration achieved the highest average ROC-AUC score.

# Manual Learning Curve

In this task, the best pipeline obtained from GridSearchCV is trained using different fractions of the training data.

The training ROC-AUC and testing ROC-AUC are compared to understand whether the model is limited by data or model capacity.

In [45]:
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

learning_results = []

In [46]:
for fraction in fractions:

    rows = int(fraction * len(X_train))

    X_subset = X_train.iloc[:rows]
    y_subset = y_train.iloc[:rows]

    best_pipeline.fit(
        X_subset,
        y_subset
    )

    train_probability = best_pipeline.predict_proba(X_subset)[:, 1]

    test_probability = best_pipeline.predict_proba(X_test)[:, 1]

    train_auc = roc_auc_score(
        y_subset,
        train_probability
    )

    test_auc = roc_auc_score(
        y_test,
        test_probability
    )

    learning_results.append({
        "Training Fraction": fraction,
        "Training AUC": train_auc,
        "Testing AUC": test_auc
    })

In [47]:
learning_curve = pd.DataFrame(
    learning_results
)

learning_curve

,Training Fraction,Training AUC,Testing AUC
0,0.2,1.000000,0.979640
1,0.4,1.000000,0.979546
2,0.6,1.000000,0.977857
3,0.8,0.999991,0.981329
4,1.0,0.999935,0.980625


# Save and Reload the Best Model

The best pipeline obtained from GridSearchCV is saved as a `.pkl` file using Joblib. The saved model is then loaded again and used for prediction.

In [48]:
import joblib

joblib.dump(best_pipeline, "best_model.pkl")

print("Model saved successfully.")

Model saved successfully.


In [49]:
loaded_model = joblib.load("best_model.pkl")

print("Model loaded successfully.")

Model loaded successfully.


In [50]:
sample_data = X_test.iloc[:2]

predictions = loaded_model.predict(sample_data)

print("Predictions")

print(predictions)

Predictions
[0 0]


### Observation

The best machine learning pipeline was successfully saved as a `.pkl` file and loaded again using Joblib.

The loaded model produced predictions without any errors, confirming that the serialization and deserialization process worked correctly.